# 12-01 LLM 理论八股

**高频面试题**: Transformer、Attention、位置编码、Tokenization、KV Cache、涌现、幻觉

---

In [ ]:
# Q1: Self-Attention 计算流程
import numpy as np
np.random.seed(42)

print("""
Q: 请描述 Self-Attention 的完整计算流程？

A: 
  1. 输入 X (seq_len × d_model)
  2. 线性变换得到 Q, K, V:
     Q = X @ W_Q,  K = X @ W_K,  V = X @ W_V
  3. 计算注意力分数:
     scores = Q @ K^T / sqrt(d_k)
  4. Softmax 归一化:
     weights = softmax(scores)   # (seq_len × seq_len)
  5. 加权求和:
     output = weights @ V        # (seq_len × d_v)

  复杂度: O(n² · d)，n=序列长度，d=维度
""")

# 代码演示
d_model, seq_len, d_k = 8, 4, 4
X = np.random.randn(seq_len, d_model).astype(np.float32)
W_Q = np.random.randn(d_model, d_k).astype(np.float32)
W_K = np.random.randn(d_model, d_k).astype(np.float32)
W_V = np.random.randn(d_model, d_k).astype(np.float32)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T / np.sqrt(d_k)
weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)  # softmax
output = weights @ V

print(f"Input X:      {X.shape}")
print(f"Q, K, V:      {Q.shape}")
print(f"Scores:       {scores.shape}")
print(f"Weights:      {weights.shape}")
print(f"Output:       {output.shape}")
print(f"\nAttention weights (每行和为1):\n{weights.round(3)}")

In [ ]:
# Q2: Multi-Head Attention
print("""
Q: 为什么要用 Multi-Head Attention？每个 head 学到什么？

A:
  Multi-Head: 将 Q/K/V 拆成 h 个 head，每个 head 独立做 attention，最后拼接
  
  MultiHead(Q,K,V) = Concat(head_1, ..., head_h) @ W_O
  head_i = Attention(Q @ W_Q_i, K @ W_K_i, V @ W_V_i)
  
  为什么多头:
  1. 不同 head 关注不同模式 (语法关系、语义关系、位置关系)
  2. 增加表达能力，类似 CNN 的多个 filter
  3. 计算量不变: h 个 head，每个 d_k = d_model/h
  
  典型配置:
  - GPT-3: d_model=12288, h=96, d_k=128
  - LLaMA-7B: d_model=4096, h=32, d_k=128

Q: 为什么除以 sqrt(d_k)？

A:
  防止点积值过大 → softmax 梯度消失
  Q·K 的方差 ≈ d_k，除以 sqrt(d_k) 后方差 ≈ 1
  没有缩放 → softmax 输出接近 one-hot → 梯度接近 0
""")

In [ ]:
# Q3: 位置编码
print("""
Q: Transformer 为什么需要位置编码？RoPE 的原理？

A:
  Self-Attention 是置换不变的 (permutation invariant)
  → "我爱你" 和 "你爱我" 的 attention 相同
  → 必须注入位置信息
  
  方案1: 正弦位置编码 (原始 Transformer)
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    - 优点: 可外推到任意长度
    - 缺点: 绝对位置，不能很好表达相对关系
  
  方案2: 可学习位置编码 (BERT/GPT)
    - 直接学一个 pos_embedding 矩阵
    - 缺点: 不能超过训练时的最大长度
  
  方案3: RoPE 旋转位置编码 (LLaMA/Qwen/主流)
    - 核心: 用旋转矩阵编码相对位置
    - q_m · k_n 只依赖于 (m-n)，天然表达相对位置
    - 公式: f(q, m) = q · e^(imθ) (复数旋转)
    - 优点: 相对位置、外推性好、计算高效
    
  方案4: ALiBi (BLOOM)
    - 不修改 embedding，直接在 attention score 上加偏置
    - bias(i,j) = -m · |i-j|, m 是 head 特定的斜率
""")

# 正弦位置编码演示
d_model, max_len = 64, 50
pe = np.zeros((max_len, d_model))
position = np.arange(max_len)[:, np.newaxis]
div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

pe[:, 0::2] = np.sin(position * div_term)
pe[:, 1::2] = np.cos(position * div_term)

print(f"位置编码矩阵: {pe.shape}")
print(f"位置0的编码 (前8维): {pe[0, :8].round(3)}")
print(f"位置1的编码 (前8维): {pe[1, :8].round(3)}")

In [ ]:
# Q4: KV Cache
print("""
Q: 什么是 KV Cache？为什么能加速推理？

A:
  自回归生成: 每个 token 依赖前面所有 token
  
  无 KV Cache (暴力计算):
    生成 token_n 时，对所有 n 个 token 重新算 Q/K/V
    总计算量: O(1² + 2² + ... + n²) = O(n³)
  
  有 KV Cache:
    缓存之前算过的 K, V
    生成 token_n 时，只算新 token 的 Q/K/V，K_cache.append(K_new)
    总计算量: O(1 + 2 + ... + n) = O(n²)
    
  代价: 显存占用增加
    KV Cache 大小 = 2 × n_layers × n_heads × seq_len × d_head × dtype_size
    LLaMA-7B, seq=2048: ≈ 2GB KV Cache
    
  优化:
    - GQA (Grouped Query Attention): 多个 Q head 共享 K/V head
      LLaMA-2-70B: 64 Q heads, 8 KV heads → KV Cache 缩小 8x
    - MQA (Multi-Query Attention): 所有 Q head 共享 1 组 KV
    - PagedAttention (vLLM): 分页管理 KV Cache，减少显存碎片
""")

# KV Cache 模拟
class KVCacheDemo:
    def __init__(self, d_model=8):
        self.k_cache = []
        self.v_cache = []
        self.d_model = d_model
        self.compute_count = 0
    
    def generate_step(self, new_token_embed):
        """模拟一步生成"""
        q_new = new_token_embed  # 简化
        k_new = new_token_embed
        v_new = new_token_embed
        
        self.k_cache.append(k_new)
        self.v_cache.append(v_new)
        
        # 只用新 Q 和全部 cached K/V 计算
        K = np.array(self.k_cache)
        V = np.array(self.v_cache)
        scores = q_new @ K.T / np.sqrt(self.d_model)
        self.compute_count += len(self.k_cache)  # 计算量
        
        return len(self.k_cache)

cache = KVCacheDemo()
print("=== KV Cache 模拟 ===")
for i in range(6):
    token = np.random.randn(8)
    seq_len = cache.generate_step(token)
    print(f"  Step {i+1}: cache_len={seq_len}, 累计计算={cache.compute_count}")

In [ ]:
# Q5: Tokenization + 涌现 + 幻觉
print("""
Q: BPE Tokenization 的原理？

A:
  1. 初始词表 = 所有单字符 (byte level: 256 个)
  2. 统计相邻 token 对出现频率
  3. 合并最高频的 pair 为新 token
  4. 重复直到词表达到目标大小 (GPT-4: ~100K)
  
  例子: "low lower lowest"
    初始: ['l','o','w',' ','l','o','w','e','r',' ','l','o','w','e','s','t']
    合并1: 'lo' (最高频 pair)
    合并2: 'low' 
    ...

Q: 什么是涌现能力 (Emergent Abilities)？

A:
  - 在小模型上不存在，大模型上突然出现的能力
  - 例子: 思维链(CoT)推理、代码生成、多语言翻译
  - 争议: 可能是评估指标不连续导致的假象 (Stanford 论文)
  - 实际应用: 大模型选择的依据之一

Q: LLM 幻觉的原因和缓解方法？

A:
  原因:
  1. 训练数据有噪声/过时信息
  2. 解码时采样随机性 (temperature > 0)
  3. 模型学的是"看起来像"而非"事实正确"
  4. 训练目标是 next token prediction，不是事实验证
  
  缓解:
  1. RAG: 检索真实文档作为 context
  2. 降低 temperature (更确定性)
  3. Self-Consistency: 多次生成取多数
  4. 事实验证工具: 让 Agent 调用知识库/搜索引擎
  5. RLHF/DPO: 训练模型说"我不确定"
  6. 结构化输出: 限制输出格式减少自由发挥
""")

## 速查表

| 题目 | 一句话答案 |
|------|-----------|
| Attention 计算 | softmax(QK^T/√d_k) × V，复杂度 O(n²d) |
| 为什么多头 | 不同 head 学不同模式，总计算量不变 |
| 为什么除 √d_k | 防止点积过大导致 softmax 梯度消失 |
| RoPE 原理 | 旋转矩阵编码相对位置，q·k 只依赖 (m-n) |
| KV Cache | 缓存历史 K/V 避免重复计算，O(n³)→O(n²) |
| GQA | 多个 Q head 共享 KV head，减少 KV Cache |
| BPE | 贪心合并高频字符对，构建子词词表 |
| 涌现 | 大模型突然出现小模型没有的能力 |
| 幻觉缓解 | RAG + 低温度 + Self-Consistency + RLHF |